# E-Commerce Customer & Revenue Analytics

## SQL Business Analysis

### Objective

This analysis uses SQL to evaluate the company's core commercial performance, including revenue, orders, average order value, customer purchasing behavior, product performance, and geographic market performance.

The goal is to translate transaction-level data into business KPIs and actionable insights.

In [3]:
!ls -lh /content

total 13M
-rw-r--r-- 1 root root  12M Sep 25 19:16 cleaned_sales.csv
drwxr-xr-x 1 root root 4.0K Sep 16 13:26 sample_data


In [4]:
!pip install -q duckdb

In [5]:
import duckdb
import pandas as pd
import numpy as np

In [6]:
con = duckdb.connect()

## 1. Load Analysis Dataset

In [7]:
con.execute("""
CREATE OR REPLACE VIEW sales AS
SELECT *
FROM read_csv_auto(
    '/content/cleaned_sales.csv',
    HEADER = TRUE
);
""")

In [8]:
con.execute("""
SELECT *
FROM sales
LIMIT 10;
""").df()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,SourceYear,IsCancelled,Year,Month,YearMonth,DayOfWeek,Hour,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-2010,False,2009,12,2009-12,Tuesday,7,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010,False,2009,12,2009-12,Tuesday,7,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010,False,2009,12,2009-12,Tuesday,7,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-2010,False,2009,12,2009-12,Tuesday,7,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-2010,False,2009,12,2009-12,Tuesday,7,30.0
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,2009-2010,False,2009,12,2009-12,Tuesday,7,39.6
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-2010,False,2009,12,2009-12,Tuesday,7,30.0
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,2009-2010,False,2009,12,2009-12,Tuesday,7,59.5
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom,2009-2010,False,2009,12,2009-12,Tuesday,7,30.6
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom,2009-2010,False,2009,12,2009-12,Tuesday,7,45.0


In [9]:
con.execute("""
SELECT COUNT(*) AS total_rows
FROM sales;
""").df()

,total_rows
0,1029609


In [10]:
con.execute("""
DESCRIBE sales;
""").df()

,column_name,column_type,null,key,default,extra
0,InvoiceNo,BIGINT,YES,None,None,None
1,StockCode,VARCHAR,YES,None,None,None
2,Description,VARCHAR,YES,None,None,None
3,Quantity,BIGINT,YES,None,None,None
4,InvoiceDate,TIMESTAMP,YES,None,None,None
5,UnitPrice,DOUBLE,YES,None,None,None
6,CustomerID,DOUBLE,YES,None,None,None
7,Country,VARCHAR,YES,None,None,None
8,SourceYear,VARCHAR,YES,None,None,None
9,IsCancelled,BOOLEAN,YES,None,None,None


## 2. Executive KPI Overview

In [13]:
con.execute("""
DROP VIEW IF EXISTS sales;
""")

In [14]:
con.execute("""
CREATE OR REPLACE VIEW sales AS

SELECT
    CAST(InvoiceNo AS VARCHAR) AS InvoiceNo,
    CAST(StockCode AS VARCHAR) AS StockCode,
    Description,

    TRY_CAST(Quantity AS DOUBLE) AS Quantity,
    TRY_CAST(InvoiceDate AS TIMESTAMP) AS InvoiceDate,
    TRY_CAST(UnitPrice AS DOUBLE) AS UnitPrice,

    CAST(CustomerID AS VARCHAR) AS CustomerID,
    Country,
    SourceYear,

    TRY_CAST(Revenue AS DOUBLE) AS Revenue

FROM read_csv_auto(
    '/content/cleaned_sales.csv',
    HEADER = TRUE,
    ALL_VARCHAR = TRUE
);
""")

In [15]:
con.execute("""
DESCRIBE sales;
""").df()

,column_name,column_type,null,key,default,extra
0,InvoiceNo,VARCHAR,YES,None,None,None
1,StockCode,VARCHAR,YES,None,None,None
2,Description,VARCHAR,YES,None,None,None
3,Quantity,DOUBLE,YES,None,None,None
4,InvoiceDate,TIMESTAMP,YES,None,None,None
5,UnitPrice,DOUBLE,YES,None,None,None
6,CustomerID,VARCHAR,YES,None,None,None
7,Country,VARCHAR,YES,None,None,None
8,SourceYear,VARCHAR,YES,None,None,None
9,Revenue,DOUBLE,YES,None,None,None


In [16]:
con.execute("""
SELECT *
FROM sales
LIMIT 10;
""").df()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,SourceYear,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12.0,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-2010,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12.0,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12.0,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48.0,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-2010,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24.0,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-2010,30.0
5,489434,22064,PINK DOUGHNUT TRINKET POT,24.0,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,2009-2010,39.6
6,489434,21871,SAVE THE PLANET MUG,24.0,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-2010,30.0
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10.0,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,2009-2010,59.5
8,489435,22350,CAT BOWL,12.0,2009-12-01 07:46:00,2.55,13085.0,United Kingdom,2009-2010,30.6
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12.0,2009-12-01 07:46:00,3.75,13085.0,United Kingdom,2009-2010,45.0


In [17]:
con.execute("""
SELECT COUNT(*) AS Total_Rows
FROM sales;
""").df()

,Total_Rows
0,1029609


In [18]:
kpi_summary = con.execute("""
SELECT
    ROUND(SUM(Revenue), 2) AS Total_Revenue,

    COUNT(DISTINCT InvoiceNo) AS Total_Orders,

    COUNT(DISTINCT CustomerID) AS Unique_Customers,

    SUM(Quantity) AS Units_Sold,

    ROUND(
        SUM(Revenue) /
        NULLIF(COUNT(DISTINCT InvoiceNo), 0),
        2
    ) AS Average_Order_Value,

    ROUND(
        SUM(Revenue) /
        NULLIF(COUNT(DISTINCT CustomerID), 0),
        2
    ) AS Revenue_Per_Customer

FROM sales;
""").df()

kpi_summary

,Total_Revenue,Total_Orders,Unique_Customers,Units_Sold,Average_Order_Value,Revenue_Per_Customer
0,20913891.47,40077,5878,11385368.0,521.84,3557.99


## 3. Monthly Revenue and Order Trends

This section analyzes monthly revenue, order volume, customer activity, units sold, and average order value to understand overall business trends.

In [19]:
monthly_performance = con.execute("""
SELECT
    DATE_TRUNC('month', InvoiceDate) AS Month,

    ROUND(SUM(Revenue), 2) AS Revenue,

    COUNT(DISTINCT InvoiceNo) AS Orders,

    COUNT(DISTINCT CustomerID) AS Customers,

    SUM(Quantity) AS Units_Sold,

    ROUND(
        SUM(Revenue) /
        NULLIF(COUNT(DISTINCT InvoiceNo), 0),
        2
    ) AS Average_Order_Value

FROM sales

GROUP BY 1

ORDER BY 1;
""").df()

monthly_performance

,Month,Revenue,Orders,Customers,Units_Sold,Average_Order_Value
0,2009-12-01,822483.95,1682,955,425461.0,488.99
1,2010-01-01,651155.11,1105,720,390672.0,589.28
2,2010-02-01,551504.73,1201,772,381879.0,459.20
3,2010-03-01,830915.26,1681,1057,526026.0,494.30
4,2010-04-01,678875.25,1462,942,366705.0,464.35
5,2010-05-01,657705.50,1500,966,395868.0,438.47
6,2010-06-01,749537.31,1645,1041,406820.0,455.65
7,2010-07-01,648810.27,1529,928,337895.0,424.34
8,2010-08-01,695251.91,1425,911,472380.0,487.90
9,2010-09-01,921696.99,1839,1145,583864.0,501.19


## 4. Month-over-Month Revenue Growth

A window function is used to compare each month's revenue with the previous month and calculate month-over-month growth.

In [20]:
mom_growth = con.execute("""
WITH monthly_sales AS (

    SELECT
        DATE_TRUNC('month', InvoiceDate) AS Month,
        SUM(Revenue) AS Revenue

    FROM sales

    GROUP BY 1
),

growth_analysis AS (

    SELECT
        Month,
        Revenue,

        LAG(Revenue) OVER (
            ORDER BY Month
        ) AS Previous_Month_Revenue

    FROM monthly_sales
)

SELECT
    Month,

    ROUND(Revenue, 2) AS Revenue,

    ROUND(
        Previous_Month_Revenue,
        2
    ) AS Previous_Month_Revenue,

    ROUND(
        (Revenue - Previous_Month_Revenue)
        / NULLIF(Previous_Month_Revenue, 0)
        * 100,
        2
    ) AS MoM_Growth_Pct

FROM growth_analysis

ORDER BY Month;
""").df()

mom_growth

,Month,Revenue,Previous_Month_Revenue,MoM_Growth_Pct
0,2009-12-01,822483.95,NaN,NaN
1,2010-01-01,651155.11,822483.95,-20.83
2,2010-02-01,551504.73,651155.11,-15.30
3,2010-03-01,830915.26,551504.73,50.66
4,2010-04-01,678875.25,830915.26,-18.30
5,2010-05-01,657705.50,678875.25,-3.12
6,2010-06-01,749537.31,657705.50,13.96
7,2010-07-01,648810.27,749537.31,-13.44
8,2010-08-01,695251.91,648810.27,7.16
9,2010-09-01,921696.99,695251.91,32.57


In [21]:
top_revenue_months = con.execute("""
SELECT
    DATE_TRUNC('month', InvoiceDate) AS Month,

    ROUND(
        SUM(Revenue),
        2
    ) AS Revenue,

    COUNT(DISTINCT InvoiceNo) AS Orders,

    COUNT(DISTINCT CustomerID) AS Customers

FROM sales

GROUP BY 1

ORDER BY Revenue DESC

LIMIT 5;
""").df()

top_revenue_months

,Month,Revenue,Orders,Customers
0,2011-11-01,1503866.78,2769,1664
1,2010-11-01,1464293.14,2747,1607
2,2010-12-01,1259083.75,1559,885
3,2010-10-01,1161902.22,2301,1497
4,2011-10-01,1151263.73,2040,1364


## 5. Geographic Market Performance

This section compares revenue, orders, customers, average order value, and revenue per customer across geographic markets.

In [22]:
country_performance = con.execute("""
WITH country_metrics AS (

    SELECT
        Country,

        SUM(Revenue) AS Revenue,

        COUNT(DISTINCT InvoiceNo) AS Orders,

        COUNT(DISTINCT CustomerID) AS Customers,

        SUM(Quantity) AS Units_Sold

    FROM sales

    GROUP BY Country
)

SELECT
    Country,

    ROUND(
        Revenue,
        2
    ) AS Revenue,

    Orders,

    Customers,

    Units_Sold,

    ROUND(
        Revenue /
        NULLIF(Orders, 0),
        2
    ) AS Average_Order_Value,

    ROUND(
        Revenue /
        NULLIF(Customers, 0),
        2
    ) AS Revenue_Per_Customer,

    ROUND(
        Revenue /
        SUM(Revenue) OVER() * 100,
        2
    ) AS Revenue_Share_Pct,

    RANK() OVER (
        ORDER BY Revenue DESC
    ) AS Revenue_Rank

FROM country_metrics

ORDER BY Revenue DESC;
""").df()

country_performance.head(15)

,Country,Revenue,Orders,Customers,Units_Sold,Average_Order_Value,Revenue_Per_Customer,Revenue_Share_Pct,Revenue_Rank
0,United Kingdom,17814055.93,36535,5350,9349186.0,487.59,3329.73,85.18,1
1,EIRE,664050.09,626,5,340054.0,1060.78,132810.02,3.18,2
2,Netherlands,554230.69,228,22,383976.0,2430.84,25192.30,2.65,3
3,Germany,430703.79,789,107,227769.0,545.89,4025.27,2.06,4
4,France,356746.51,622,95,275090.0,573.55,3755.23,1.71,5
5,Australia,169900.61,95,15,104080.0,1788.43,11326.71,0.81,6
6,Spain,109127.21,154,41,50774.0,708.62,2661.64,0.52,7
7,Switzerland,100988.99,93,22,52872.0,1085.90,4590.41,0.48,8
8,Sweden,91869.82,105,19,88633.0,874.95,4835.25,0.44,9
9,Denmark,69862.19,43,12,237925.0,1624.70,5821.85,0.33,10


In [23]:
non_uk_markets = con.execute("""
SELECT
    Country,

    ROUND(
        SUM(Revenue),
        2
    ) AS Revenue,

    COUNT(DISTINCT InvoiceNo) AS Orders,

    COUNT(DISTINCT CustomerID) AS Customers,

    ROUND(
        SUM(Revenue) /
        NULLIF(COUNT(DISTINCT InvoiceNo), 0),
        2
    ) AS Average_Order_Value,

    ROUND(
        SUM(Revenue) /
        NULLIF(COUNT(DISTINCT CustomerID), 0),
        2
    ) AS Revenue_Per_Customer

FROM sales

WHERE Country <> 'United Kingdom'

GROUP BY Country

HAVING COUNT(DISTINCT InvoiceNo) >= 20

ORDER BY Revenue DESC;
""").df()

non_uk_markets.head(15)

,Country,Revenue,Orders,Customers,Average_Order_Value,Revenue_Per_Customer
0,EIRE,664050.09,626,5,1060.78,132810.02
1,Netherlands,554230.69,228,22,2430.84,25192.30
2,Germany,430703.79,789,107,545.89,4025.27
3,France,356746.51,622,95,573.55,3755.23
4,Australia,169900.61,95,15,1788.43,11326.71
5,Spain,109127.21,154,41,708.62,2661.64
6,Switzerland,100988.99,93,22,1085.90,4590.41
7,Sweden,91869.82,105,19,874.95,4835.25
8,Denmark,69862.19,43,12,1624.70,5821.85
9,Belgium,65733.92,149,29,441.17,2266.69


## 6. Product Performance

Products are evaluated by revenue, units sold, order frequency, and contribution to overall revenue.

In [24]:
product_performance = con.execute("""
WITH product_metrics AS (

    SELECT
        StockCode,
        Description,

        SUM(Revenue) AS Revenue,

        SUM(Quantity) AS Units_Sold,

        COUNT(DISTINCT InvoiceNo) AS Orders

    FROM sales

    WHERE Description IS NOT NULL

    GROUP BY
        StockCode,
        Description
)

SELECT
    StockCode,

    Description,

    ROUND(
        Revenue,
        2
    ) AS Revenue,

    Units_Sold,

    Orders,

    ROUND(
        Revenue /
        NULLIF(Orders, 0),
        2
    ) AS Revenue_Per_Order,

    ROUND(
        Revenue /
        SUM(Revenue) OVER() * 100,
        2
    ) AS Revenue_Share_Pct,

    RANK() OVER (
        ORDER BY Revenue DESC
    ) AS Revenue_Rank

FROM product_metrics

ORDER BY Revenue DESC;
""").df()

product_performance.head(20)

,StockCode,Description,Revenue,Units_Sold,Orders,Revenue_Per_Order,Revenue_Share_Pct,Revenue_Rank
0,22423,REGENCY CAKESTAND 3 TIER,344069.30,27536.0,3918,87.82,1.65,1
1,M,Manual,340327.63,9748.0,784,434.09,1.63,2
2,DOT,DOTCOM POSTAGE,322657.48,1436.0,1415,228.03,1.54,3
3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,262589.96,95966.0,5356,49.03,1.26,4
4,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995.0,1,168469.60,0.81,5
5,47566,PARTY BUNTING,149109.35,28362.0,2674,55.76,0.71,6
6,85099B,JUMBO BAG RED RETROSPOT,148528.63,78698.0,3245,45.77,0.71,7
7,84879,ASSORTED COLOUR BIRD ORNAMENT,131817.81,81590.0,2807,46.96,0.63,8
8,POST,POSTAGE,127597.42,5461.0,1851,68.93,0.61,9
9,22086,PAPER CHAIN KIT 50'S CHRISTMAS,123002.89,36534.0,2018,60.95,0.59,10


In [25]:
top_products_quantity = con.execute("""
SELECT
    StockCode,

    Description,

    SUM(Quantity) AS Units_Sold,

    ROUND(
        SUM(Revenue),
        2
    ) AS Revenue,

    COUNT(DISTINCT InvoiceNo) AS Orders

FROM sales

WHERE Description IS NOT NULL

GROUP BY
    StockCode,
    Description

ORDER BY Units_Sold DESC

LIMIT 20;
""").df()

top_products_quantity

,StockCode,Description,Units_Sold,Revenue,Orders
0,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,109898.0,25197.98,1019
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,95966.0,262589.96,5356
2,84879,ASSORTED COLOUR BIRD ORNAMENT,81590.0,131817.81,2807
3,23843,"PAPER CRAFT , LITTLE BIRDIE",80995.0,168469.60,1
4,85099B,JUMBO BAG RED RETROSPOT,78698.0,148528.63,3245
5,23166,MEDIUM CERAMIC TOP STORAGE JAR,78033.0,81700.92,247
6,17003,BROCADE RING PURSE,71394.0,14948.75,456
7,21977,PACK OF 60 PINK PAISLEY CAKE CASES,56625.0,28413.63,1993
8,84991,60 TEATIME FAIRY CAKE CASES,54537.0,27321.37,2127
9,22197,SMALL POPCORN HOLDER,49922.0,44068.03,1407


## 7. Customer Purchasing Behavior

Customer-level metrics are calculated to evaluate purchase frequency, spending, and repeat purchasing behavior.

In [26]:
customer_metrics = con.execute("""
SELECT
    CustomerID,

    COUNT(DISTINCT InvoiceNo) AS Orders,

    ROUND(
        SUM(Revenue),
        2
    ) AS Revenue,

    SUM(Quantity) AS Units_Purchased,

    MIN(InvoiceDate) AS First_Purchase,

    MAX(InvoiceDate) AS Last_Purchase,

    ROUND(
        SUM(Revenue) /
        NULLIF(COUNT(DISTINCT InvoiceNo), 0),
        2
    ) AS Average_Order_Value

FROM sales

WHERE CustomerID IS NOT NULL

GROUP BY CustomerID

ORDER BY Revenue DESC;
""").df()

customer_metrics.head(20)

,CustomerID,Orders,Revenue,Units_Purchased,First_Purchase,Last_Purchase,Average_Order_Value
0,18102.0,145,608821.65,188340.0,2009-12-01 09:24:00,2011-12-09 11:50:00,4198.77
1,14646.0,151,528602.52,367193.0,2009-12-02 16:52:00,2011-12-08 12:12:00,3500.68
2,14156.0,156,313759.82,165873.0,2009-12-01 12:30:00,2011-11-30 10:54:00,2011.28
3,14911.0,398,295832.39,149949.0,2009-12-01 11:41:00,2011-12-08 15:54:00,743.30
4,17450.0,51,246813.09,84700.0,2010-09-27 16:59:00,2011-12-01 13:29:00,4839.47
5,13694.0,143,196482.81,189205.0,2009-12-04 15:26:00,2011-12-06 09:32:00,1374.01
6,17511.0,60,175603.55,119656.0,2009-12-02 10:52:00,2011-12-07 10:12:00,2926.73
7,16446.0,2,168472.50,80997.0,2011-05-18 09:52:00,2011-12-09 09:15:00,84236.25
8,16684.0,55,147142.77,104810.0,2009-12-07 12:56:00,2011-12-05 14:06:00,2675.32
9,12415.0,28,144458.37,91447.0,2010-06-30 08:30:00,2011-11-15 14:22:00,5159.23


In [27]:
repeat_customer_summary = con.execute("""
WITH customer_orders AS (

    SELECT
        CustomerID,

        COUNT(
            DISTINCT InvoiceNo
        ) AS Order_Count

    FROM sales

    WHERE CustomerID IS NOT NULL

    GROUP BY CustomerID
)

SELECT
    COUNT(*) AS Total_Customers,

    SUM(
        CASE
            WHEN Order_Count > 1
            THEN 1
            ELSE 0
        END
    ) AS Repeat_Customers,

    SUM(
        CASE
            WHEN Order_Count = 1
            THEN 1
            ELSE 0
        END
    ) AS One_Time_Customers,

    ROUND(
        SUM(
            CASE
                WHEN Order_Count > 1
                THEN 1
                ELSE 0
            END
        ) * 100.0
        / COUNT(*),
        2
    ) AS Repeat_Customer_Rate

FROM customer_orders;
""").df()

repeat_customer_summary

,Total_Customers,Repeat_Customers,One_Time_Customers,Repeat_Customer_Rate
0,5878,4255.0,1623.0,72.39


In [28]:
date_range = con.execute("""
SELECT
    MIN(InvoiceDate) AS Start_Date,
    MAX(InvoiceDate) AS End_Date
FROM sales;
""").df()

date_range

,Start_Date,End_Date
0,2009-12-01 07:45:00,2011-12-09 12:50:00


In [29]:
customer_id_coverage = con.execute("""
SELECT
    COUNT(*) AS Total_Rows,

    SUM(
        CASE
            WHEN CustomerID IS NULL THEN 1
            ELSE 0
        END
    ) AS Missing_Customer_Rows,

    ROUND(
        SUM(
            CASE
                WHEN CustomerID IS NULL THEN Revenue
                ELSE 0
            END
        ),
        2
    ) AS Missing_Customer_Revenue,

    ROUND(
        SUM(
            CASE
                WHEN CustomerID IS NULL THEN Revenue
                ELSE 0
            END
        ) / SUM(Revenue) * 100,
        2
    ) AS Missing_Customer_Revenue_Pct

FROM sales;
""").df()

customer_id_coverage

,Total_Rows,Missing_Customer_Rows,Missing_Customer_Revenue,Missing_Customer_Revenue_Pct
0,1029609,236000.0,3228430.83,15.44


In [30]:
kpi_summary = con.execute("""
SELECT
    ROUND(SUM(Revenue), 2) AS Total_Revenue,

    COUNT(DISTINCT InvoiceNo) AS Total_Orders,

    COUNT(DISTINCT CustomerID) AS Unique_Customers,

    SUM(Quantity) AS Units_Sold,

    ROUND(
        SUM(Revenue) /
        NULLIF(COUNT(DISTINCT InvoiceNo), 0),
        2
    ) AS Average_Order_Value,

    ROUND(
        SUM(
            CASE
                WHEN CustomerID IS NOT NULL
                THEN Revenue
                ELSE 0
            END
        ) /
        NULLIF(COUNT(DISTINCT CustomerID), 0),
        2
    ) AS Revenue_Per_Known_Customer

FROM sales;
""").df()

kpi_summary

,Total_Revenue,Total_Orders,Unique_Customers,Units_Sold,Average_Order_Value,Revenue_Per_Known_Customer
0,20913891.47,40077,5878,11385368.0,521.84,3008.75


In [31]:
top_product_check = con.execute("""
SELECT
    InvoiceNo,
    InvoiceDate,
    CustomerID,
    Country,
    StockCode,
    Description,
    Quantity,
    UnitPrice,
    Revenue

FROM sales

WHERE StockCode = '23843'

ORDER BY Revenue DESC;
""").df()

top_product_check

,InvoiceNo,InvoiceDate,CustomerID,Country,StockCode,Description,Quantity,UnitPrice,Revenue
0,581483,2011-12-09 09:15:00,16446.0,United Kingdom,23843,"PAPER CRAFT , LITTLE BIRDIE",80995.0,2.08,168469.6


In [32]:
kpi_summary.to_csv(
    "/content/kpi_summary.csv",
    index=False
)

monthly_performance.to_csv(
    "/content/monthly_performance.csv",
    index=False
)

mom_growth.to_csv(
    "/content/mom_growth.csv",
    index=False
)

country_performance.to_csv(
    "/content/country_performance.csv",
    index=False
)

product_performance.to_csv(
    "/content/product_performance.csv",
    index=False
)

customer_metrics.to_csv(
    "/content/customer_metrics.csv",
    index=False
)

repeat_customer_summary.to_csv(
    "/content/repeat_customer_summary.csv",
    index=False
)

In [33]:
!ls -lh /content/*.csv

-rw-r--r-- 1 root root 137M Sep 25 19:17 /content/cleaned_sales.csv
-rw-r--r-- 1 root root 2.4K Sep 25 19:39 /content/country_performance.csv
-rw-r--r-- 1 root root 405K Sep 25 19:39 /content/customer_metrics.csv
-rw-r--r-- 1 root root  151 Sep 25 19:39 /content/kpi_summary.csv
-rw-r--r-- 1 root root  976 Sep 25 19:39 /content/mom_growth.csv
-rw-r--r-- 1 root root 1.2K Sep 25 19:39 /content/monthly_performance.csv
-rw-r--r-- 1 root root 363K Sep 25 19:39 /content/product_performance.csv
-rw-r--r-- 1 root root   98 Sep 25 19:39 /content/repeat_customer_summary.csv
